# 12 — Multi-Agent Collaboration

Supervisor orchestrates researcher and writer agents.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

## Knowledge Base

In [ ]:
KNOWLEDGE = {
    "python": "Python is a high-level programming language created by Guido van Rossum in 1991. Key features: dynamic typing, garbage collection, multiple paradigms (OOP, functional, procedural). Popular for web dev, data science, ML, and scripting.",
    "rust": "Rust is a systems programming language focused on safety, speed, and concurrency. Created by Graydon Hoare at Mozilla, first stable release in 2015. Key features: ownership system, zero-cost abstractions, no garbage collector.",
    "go": "Go (Golang) was designed at Google by Robert Griesemer, Rob Pike, and Ken Thompson, released in 2009. Key features: static typing, garbage collection, built-in concurrency (goroutines), fast compilation.",
}

## Worker Agents

In [ ]:
def researcher_agent(topic: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    context = "\n".join(info for key, info in KNOWLEDGE.items() if key in topic.lower()) or "Use general knowledge."
    prompt = ChatPromptTemplate.from_template(
        "You are a research assistant. Provide key facts about this topic.\n\nContext:\n{context}\n\nTopic: {topic}\n\nResearch notes:"
    )
    return (prompt | llm | StrOutputParser()).invoke({"topic": topic, "context": context})

def writer_agent(topic: str, research: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    prompt = ChatPromptTemplate.from_template(
        "You are a technical writer. Write a concise summary (3-4 sentences).\n\nTopic: {topic}\n\nResearch notes:\n{research}\n\nSummary:"
    )
    return (prompt | llm | StrOutputParser()).invoke({"topic": topic, "research": research})

## Supervisor Orchestration

In [ ]:
def supervisor_agent(user_request: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print(f"  [Supervisor] Assigning research...")
    research = researcher_agent(user_request)
    print(f"  [Researcher] Done ({len(research)} chars)")
    print(f"  [Supervisor] Assigning writing...")
    draft = writer_agent(user_request, research)
    print(f"  [Writer] Done ({len(draft)} chars)")
    review_chain = ChatPromptTemplate.from_template(
        "Review and polish this draft. Keep it concise.\n\nRequest: {request}\nDraft:\n{draft}\n\nFinal version:"
    ) | llm | StrOutputParser()
    return review_chain.invoke({"request": user_request, "draft": draft})

for task in ["Write a brief overview of Python as a programming language", "Compare Rust and Go for systems programming"]:
    print(f"\nRequest: {task}")
    result = supervisor_agent(task)
    print(f"\nFinal output:\n{result}\n" + "-"*60)